# fedstat — проверка и примеры

Демонстрация библиотеки [`fedstat`](https://pypi.org/project/fedstat/) на реальных показателях ЕМИСС.
Каждый пример: посмотреть доступные фильтры → задать нужные → скачать `DataFrame` → «широкий» вид.

> Требуется доступ к fedstat.ru (российский IP). Установка: `pip install -U fedstat`.

In [ ]:
import fedstat
import pandas as pd

## Как узнать, какие фильтры доступны

`filter_options(id)` — обзор всех полей и их уникальных значений: словарём `{поле: [значения]}` или таблицей (`as_frame=True`). Заменяет ручной перебор через `.unique()`.

In [ ]:
fedstat.filter_options("31452", as_frame=True)

In [ ]:
# только конкретное поле:
fedstat.filter_options("31452")["Типы квартир"]

> **Про «широкий» вид (`to_wide`).** Это `pivot_table` со средним по умолчанию. В таблицу попадают только `index` и `columns`; **все прочие измерения (регион, рынок и т.п.) усредняются**. Поэтому для осмысленного ряда сначала фильтруем до одного региона (ниже — по РФ). Чтобы оставить все регионы, добавь их в `index`: `to_wide(df, columns=..., index=["region", "year"])`.

## Пример 1 — Средняя цена 1 кв. м жилья (31452)

Кварталы, по годам и рынкам.

In [ ]:
f = fedstat.filter_template("31452")
f["Год"] = [str(y) for y in range(2017, 2027)]
f["Рынок жилья"] = ["Первичный рынок жилья", "Вторичный рынок жилья"]
f["Типы квартир"] = ["Все типы квартир"]

OKATO = "Классификатор объектов административно-территориального деления (ОКАТО)"
df_price = (
    fedstat.load("31452", filters=f)
    .rename(columns={OKATO: "region", "Рынок жилья": "market",
                     "PERIOD": "period", "TIME": "year", "VALUE": "value"})
    [["region", "market", "period", "year", "value"]]
)
df_price.head()

In [ ]:
# «широкий» вид ПО РФ, первичный рынок (иначе усреднится по всем регионам)
rf = df_price[(df_price.region.isin(["Российская Федерация", "Российская Федерация без учета новых субъектов (с 01.01.2023)"])) & (df_price.market == "Первичный рынок жилья")]
fedstat.to_wide(rf, columns="period", values="value", index="year")

,year,I квартал,II квартал,III квартал,IV квартал
0,2017,56347.2,56516.78,56560.78,56882.19
1,2018,58875.59,59969.66,60952.83,61831.57
2,2019,60705.14,61618.25,62891.94,64059.49
3,2020,71503.24,73438.05,76167.22,79003.0
4,2021,83177.29,89007.64,93536.96,98908.96
5,2022,109197.56,116278.85,121315.13,122342.88
6,2023,127228.93,128828.73,134098.13,140370.82
7,2024,167580.49,171165.59,175075.58,177887.3
8,2025,201954.59,205097.71,208267.82,215282.24
9,2026,219521.77,221973.62,<NA>,<NA>


In [ ]:
fedstat.to_wide(df_price, columns="period", values="value", index=["region","market","year"])

,region,market,year,I квартал,II квартал,III квартал,IV квартал
0,Алтайский край,Вторичный рынок жилья,2017,41054.31,40507.41,41104.68,41292.57
1,Алтайский край,Вторичный рынок жилья,2018,42311.22,42565.36,43213.57,44219.05
2,Алтайский край,Вторичный рынок жилья,2019,45565.59,46272.93,46868.43,47753.73
3,Алтайский край,Вторичный рынок жилья,2020,48204.72,47336.55,50722.45,53520.0
4,Алтайский край,Вторичный рынок жилья,2021,57637.39,59313.8,61317.89,63856.0
...,...,...,...,...,...,...,...
1874,Ярославская область,Первичный рынок жилья,2022,75123.04,81167.94,83763.47,86630.32
1875,Ярославская область,Первичный рынок жилья,2023,89312.69,87003.33,94078.44,95040.77
1876,Ярославская область,Первичный рынок жилья,2024,103400.21,105831.33,106236.91,107824.96
1877,Ярославская область,Первичный рынок жилья,2025,102032.82,103536.27,105978.82,109624.05


## Пример 2 — Индекс цен на жильё (30925)

Есть поле «Виды показателя» — выбираем один вид.

In [ ]:
fedstat.filter_options("30925")["Виды показателя"]

In [ ]:
f = fedstat.filter_template("30925")
f["Год"] = [str(y) for y in range(2017, 2027)]
f["Виды показателя"] = ["К соответствующему кварталу предыдущего года"]
f["Типы квартир"] = ["Все типы квартир"]

df_hpi = (
    fedstat.load("30925", filters=f)
    .rename(columns={OKATO: "region", "Рынок жилья": "market",
                     "PERIOD": "period", "TIME": "year", "VALUE": "value"})
    [["region", "market", "period", "year", "value"]]
)
df_hpi.head()

In [ ]:
rf = df_hpi[(df_hpi.region == "Российская Федерация") & (df_hpi.market == "Первичный рынок жилья")]
fedstat.to_wide(rf, columns="period", values="value", index="year")

## Пример 3 — Уровень безработицы (43062)

Есть поле «Возраст».

In [ ]:
fedstat.filter_options("43062")["Возраст"]

In [ ]:
f = fedstat.filter_template("43062")
f["Год"] = [str(y) for y in range(2017, 2027)]
f["Возраст"] = ["15 лет и старше"]
f["Период"] = ["I квартал", "II квартал", "III квартал", "IV квартал"]

df_unemp = (
    fedstat.load("43062", filters=f)
    .rename(columns={OKATO: "region", "PERIOD": "period", "TIME": "year", "VALUE": "value"})
    [["region", "period", "year", "value"]]
)
df_unemp.head()

In [ ]:
rf = df_unemp[df_unemp.region == "Российская Федерация"]
fedstat.to_wide(rf, columns="period", values="value", index="year")

## Пример 4 — Ставки по ипотеке (59345, месячные данные)

**Месячная** периодика и иерархические регионы.

In [ ]:
fedstat.filter_options("59345")["Характеристики кредита"]

In [ ]:
f = fedstat.filter_template("59345")
f["Год"] = [str(y) for y in range(2018, 2027)]
f["Характеристики жилищных/ипотечных кредитов"] = ["Средневзвешенная ставка по кредитам, выданным в течение месяца"]
f["Характеристики кредита"] = ["Ипотечные жилищные кредиты"]

REGION = "Регионы Российской Федерации (иерархический)"
df_rate = (
    fedstat.load("59345", filters=f)
    .rename(columns={REGION: "region", "PERIOD": "month", "TIME": "year", "VALUE": "value"})
    [["region", "month", "year", "value"]]
)
df_rate.head()

In [ ]:
rf = df_rate[df_rate.region == "Российская Федерация"]
fedstat.to_wide(rf, columns="month", values="value", index="year")